# درس ۱۲ - کاهش سابقه چت با آژانس Scratchpad

این نوت‌بوک نشان می‌دهد چگونه می‌توان با استفاده از چارچوب عامل مایکروسافت، در گفتگوهای طولانی زمینه را مدیریت کرد. با افزایش گفتگوها، تعداد توکن‌ها افزایش می‌یابد — که در نهایت از پنجره زمینه مدل فراتر می‌رود. ما این موضوع را با **الگوی خلاصه‌سازی زمینه** و یک **آژانس Scratchpad** برای حافظه پایدار حل می‌کنیم.

## آنچه خواهید آموخت:
۱. **چرا مدیریت زمینه مهم است**: درک محدودیت‌های توکن و پنجره‌های زمینه
۲. **عوامل آگاه به زمینه**: ساخت عوامل که مدیریت زمینه گفتگوی خود را بر عهده دارند
۳. **الگوی خلاصه‌سازی زمینه**: استفاده از ابزارها برای کاهش تاریخچه گفتگو
۴. **آژانس Scratchpad**: حافظه پایدار که پس از کاهش زمینه نیز حفظ می‌شود

## پیش‌نیازها:
- راه‌اندازی Azure OpenAI با پیکربندی متغیرهای محیطی
- درک مفاهیم پایه عامل از درس‌های قبلی


## راه‌اندازی


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## چرا مدیریت زمینه اهمیت دارد

هر مدل زبان بزرگ (LLM) یک **پنجره زمینه‌ای** محدود دارد — بیشینه تعداد توکنی که می‌تواند در یک درخواست پردازش کند. با پیشرفت یک گفتگوی چندمرحله‌ای:

- **تعداد توکن‌ها به صورت خطی افزایش می‌یابد** با هر پیغام کاربر و پاسخ دستیار.
- **توکن‌های پرامپت هزینه را تعیین می‌کنند** چون کل تاریخچه در هر دور دوباره ارسال می‌شود.
- در نهایت گفتگو **از پنجره زمینه‌ای فراتر می‌رود** و مدل یا کوتاه می‌کند یا خطا می‌دهد.

### راهکارهای مدیریت زمینه

| راهکار | نحوه کارکرد | تبادل |
|---|---|---|
| **کوتاه‌سازی** | حذف قدیمی‌ترین پیام‌ها | از دست رفتن زمینه اولیه |
| **خلاصه‌سازی** | فشرده‌کردن پیام‌های قدیمی‌تر به یک خلاصه | مقداری جزئیات از دست می‌رود، اما نکات کلیدی حفظ می‌شود |
| **دفترچه یادداشت / حافظه خارجی** | ذخیره حقایق کلیدی خارج از گفتگو | نیاز به فراخوانی ابزار دارد، اما در هر کاهش حفظ می‌شود |

در این دفترچه یادداشت، ما **خلاصه‌سازی** را با ابزاری برای **دفترچه یادداشت** ترکیب می‌کنیم تا عامل بتواند حتی وقتی تاریخچه گفتگو فشرده می‌شود، پیوستگی را حفظ کند.


## ایجاد یک عامل آگاه به متن  


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## شبیه‌سازی یک گفتگوی طولانی

بیایید یک گفتگوی چند مرحله‌ای را مرور کنیم تا ببینیم چگونه زمینه تجمع می‌یابد. نماینده باید جزئیات کلیدی (ترجیحات، بودجه، تاریخ‌های سفر) را در طول مراحل حفظ کند و پیوستگی را نشان دهد.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

توجه کنید که چگونه عامل متن را از گردش‌های قبلی حفظ می‌کند — او دربارهٔ ژاپن، سوشی، معابد، عکاسی، بودجه ۳۰۰۰ دلاری، سفر تنها و بازه زمانی آوریل می‌داند. در یک گفتگوی کوتاه این روش خوب کار می‌کند، اما با گسترش گفتگو، ارسال مجدد کل تاریخچه هزینه‌بر می‌شود.

بیایید گفتگو را با گردش‌های بیشتر ادامه دهیم تا تجمع متن را ببینیم:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## الگوی خلاصه‌سازی زمینه

با گسترش گفتگو، می‌توان از یک **ابزار خلاصه‌سازی** برای چکیده‌کردن زمینه جمع‌آوری شده به یک قالب فشرده استفاده کرد. عامل این ابزار را فراخوانی می‌کند تا ترجیحات کلیدی را ثبت کند تا حتی اگر پیام‌های قدیمی‌تر حذف شوند، اطلاعات اساسی حفظ شود.

این الگو بلوک سازنده‌ای برای کاهش تاریخچه پیشرفته‌تر است:
1. عامل حقایق کلیدی را از گفتگو شناسایی می‌کند
2. ابزار خلاصه‌سازی را فراخوانی می‌کند تا آن‌ها را ذخیره کند
3. پیام‌های قدیمی‌تر می‌توانند با اطمینان حذف شوند زیرا خلاصه آنچه مهم است را ثبت می‌کند

در ادامه، ما ابزاری به نام `summarize_preferences` تعریف می‌کنیم که عامل می‌تواند آن را برای ثبت یک خلاصه فشرده از آنچه یاد گرفته است، فراخوانی کند.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## خلاصه

در این درس یاد گرفتید چگونه مدیریت زمینه را در مکالمات بلندمدت نمایندگان با استفاده از چارچوب Microsoft Agent انجام دهید:

### مفاهیم کلیدی
- **پنجره‌های زمینه محدود هستند** — هر توکن در تاریخچه مکالمه هزینه‌بر است و به محدودیت شمارش می‌شود.
- **ابزارهای خلاصه‌سازی** به نماینده اجازه می‌دهند تا زمینه جمع‌آوری شده را به خلاصه‌های فشرده کاهش دهد، استفاده از توکن را کاهش داده و اطلاعات ضروری را حفظ کند.
- **دفترچه‌های نماینده** حافظه خارجی پایداری فراهم می‌کنند که پس از هر کاهش مکالمه باقی می‌ماند.

### آنچه ساختید
- یک **نماینده آگاه به زمینه** که پیوستگی در مکالمات چندمرحله‌ای را حفظ می‌کند
- یک **ابزار خلاصه‌سازی** (`summarize_preferences`) که جزئیات کلیدی کاربر را به شکل فشرده ضبط می‌کند
- یک **مکالمه چندمرحله‌ای** که نگهداری زمینه و مدیریت تغییرات را نشان می‌دهد

### کاربردهای دنیای واقعی
- **ربات‌های خدمات مشتری**: به خاطر سپردن ترجیحات در جلسات طولانی پشتیبانی
- **دستیارهای شخصی**: پیگیری پروژه‌های جاری بدون نیاز به توضیح دوباره زمینه
- **معلمان آموزشی**: حفظ پیشرفت دانش‌آموز طی تعاملات متعدد

### مراحل بعدی
- پیاده‌سازی ابزار کامل دفترچه با حفظ فایل‌بندی
- افزودن کاهش خودکار تاریخچه پس از خلاصه‌سازی
- ترکیب با پایگاه‌های داده برداری برای جستجوی معنایی حافظه
- ساخت نمایندگانی که بتوانند مکالمات را روزها بعد با حفظ کامل زمینه از سر بگیرند


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
